# Enclosing radially convex set

Given a set of circles in the plane, let us find a **star-shaped (radially convex) region** that encloses all of them while minimizing a weighted sum of area and perimeter.

The enclosing shape is parameterized by its **center** $(c_x, c_y)$ and **K radii** at uniformly-spaced angles $\theta_k = 2\pi k / K$. The boundary point at angle $k$ is:
$$p_k = \text{center} + r_k \begin{pmatrix} \cos\theta_k \\ \sin\theta_k \end{pmatrix}$$

The optimization minimizes a weighted sum of enclosure violation, area, and perimeter.

The **enclosure loss** penalizes squared violations of $r_k \geq (\mathbf{c}_i - \mathbf{c}) \cdot \hat{\theta}_k + r_i$, i.e. the star boundary must extend at least as far as each circle's support in every sampled direction.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import SVG
#from jax import numpy as jnp

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import OptimConfig
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles,
)

## Allowing circles to move

In [ ]:
circles = [
    # Set 0: left cluster
    (0.0,  0.0, 0.6),
    (1.2,  0.3, 0.5),
    (0.3,  1.4, 0.4),
    # Set 1: right cluster
    (4.5,  0.0, 0.6),
    (5.5,  0.5, 0.5),
    (4.2,  1.3, 0.4),
    # Set 2: isolated circle (obstacle for both)
    (2.5,  0.5, 0.3),
    # Additional bottom circles
    (1.2,  -1.3, 0.5),
    (5.5,  -1.5, 0.5),
]

sets = [
    [0, 1, 2],     # left cluster
    [3, 4, 5],     # right cluster
    [1, 3, 7, 8],  # bottom set, partially intersecting left and right cluster
    [3, 4, 5, 8],  # right + bottom right
]

offsets = np.array([[0.1], [0.1], [0.1], [0.2]])

results, circles_out, history, problem = optimize_radially_convex_sets_and_circles(
    circles,
    sets,
    weight_area=1.0,
    weight_perimeter=2.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=1.0,
    weight_circle_collision=100.,
    offsets=offsets,
    optim_config=OptimConfig(n_iters=10000, learning_rate=0.002),
)

problem.plot(show_arrows=True)
plt.title("Optimization results: dashed = initial, solid = optimized (arrows show movement)")
plt.show()

## British Islands — joint multi-set and circle layout

The **British Islands** have a natural nested set structure:

- **Great Britain** contains England, Scotland, Wales
- **United Kingdom** contains Great Britain + Northern Ireland
- **British Islands** contains United Kingdom + Jersey, Guernsey, Isle of Man

We optimize *both* the star-shaped boundaries of each set *and* the positions of the element circles simultaneously.

## British Islands — `from_graph` API

Same problem as above, but using the graph directly — no manual extraction of circles and sets.

In [ ]:
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles_from_graph,
)
from vizopt.examples.sets import make_british_islands_graph

inclusion_graph = make_british_islands_graph(include_ireland_island=True)

# Compute per-set offsets from nesting depth (same logic as above)
set_names_fg = [n for n in nx.topological_sort(inclusion_graph) if inclusion_graph.out_degree(n) > 0]
leaf_set_fg = {n for n in inclusion_graph.nodes if inclusion_graph.out_degree(n) == 0}
nesting_depths_fg = [
    max(nx.shortest_path_length(inclusion_graph, s, leaf) for leaf in leaf_set_fg if nx.has_path(inclusion_graph, s, leaf))
    for s in set_names_fg
]
offsets_fg = np.array([[0.15 + 0.2 * d] for d in nesting_depths_fg])

snapshot_cb_fg = SnapshotCallback(every=100)

results_fg, circles_fg, history_fg, problem_fg = (
    optimize_radially_convex_sets_and_circles_from_graph(
        inclusion_graph,
        weight_area=1.0,
        weight_perimeter=2.0,
        weight_exclusion=20.0,
        weight_smoothness=3.0,
        weight_position_anchor=3.0,
        weight_circle_collision=100.0,
        weight_set_attraction=1.0,
        circle_collision_alpha=1.0,
        offsets=offsets_fg,
        optim_config=OptimConfig(n_iters=3000, learning_rate=0.002),
        callback=snapshot_cb_fg,
    )
)
print(f"Sets: {list(results_fg)}")
print(f"Circles: {list(circles_fg)}")
print(f"Captured {len(snapshot_cb_fg.snapshots)} snapshots")

In [ ]:
problem_fg.plot(show_arrows=True)
plt.title("British Islands — from_graph API\n(dashed = initial, arrows = movement)")
plt.show()

In [ ]:
svg = snapshots_to_animated_svg(
    problem_fg,
    snapshot_cb_fg.snapshots,
    fps=5,
    size=500,
    history=history_fg,
    optim_vars_panel_height=150,
)
SVG(data=svg)